# DFST Generalization on the Universal Learning Task

Plot the six selected DFST runs: dataset sizes 75, 150 and 225, each trained for 250k and 500k steps. Select runs through `DFST_universal` in `paper/figures.yaml`. Each selected directory contains the training configuration and final evaluation.

This notebook reads saved final-checkpoint evaluations from `runs/`. The notebook uses saved results without training or checkpoint evaluation. The run table lists the selected configurations and measured results. Every selected run requires a final evaluation. Zero denotes a measured result; `≥` marks a tested ceiling.

Run all cells to export one task-generalization figure per selected run, a run inventory and the metric table. Figures and tables use `analysis_outputs/paper_figures/DFST/universal/`. `DEL_ANALYSIS_DIR` selects another figure root and preserves these subfolders. Figures use the shared paper style.


In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path(os.environ["DEL_REPO_ROOT"]) if os.environ.get("DEL_REPO_ROOT") else next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/differentiable_exact_learning").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from differentiable_exact_learning.analysis.release import figure_group, final_results_table, parameter_count
from differentiable_exact_learning.analysis.paper_heatmaps import save_figure, figure_output_dir
from differentiable_exact_learning.analysis.plot_style import apply_paper_style
apply_paper_style()
OUTPUT_DIR = figure_output_dir(ROOT, 'DFST', 'universal')
def export(fig, name):
    paths = save_figure(fig, OUTPUT_DIR / name)
    display(fig)
    plt.close(fig)
    return paths

In [ ]:
from differentiable_exact_learning.analysis.universal_figures import plot_universal_generalization
MODEL = "DFST"
records = figure_group(ROOT, f"{MODEL}_universal")
if len(records) != 6:
    raise ValueError("Expected 6 selected DFST universal runs")
inventory = pd.DataFrame([{"run": row["run_id"], "dataset_size": row["config"]["data"]["size"],
                           "hidden_dimensions": row["config"]["model"]["hidden_size"],
                           "training_steps": row["config"]["training"]["steps"]} for row in records])
display(inventory)
results = final_results_table(records)
display(results)
inventory.to_csv(OUTPUT_DIR / "dfst_universal_runs.csv", index=False)


## Task Generalization

The shaded region marks the training lengths and tasks. Scaling-A and Scaling-B vary program size with input length. Other tasks vary input length. Each title identifies the controller, capacity, dataset size, and training budget.

In [ ]:
for record in records:
    c = record["config"]
    scores = {task: values["generalization"] for task, values in record["evaluation"]["tasks"].items()
              if "generalization" in values}
    h, size, steps = c["model"]["hidden_size"], c["data"]["size"], c["training"]["steps"]
    fig = plot_universal_generalization(scores, h, size, max_length=c["data"]["max_length"],
                                         parameter_count=parameter_count(record))
    fig.axes[0].set_title(f"{MODEL} · Hidden Dimensions {h} · Dataset Size {size} · Steps {steps:,}")
    export(fig, f"universal_{MODEL}_h{h}_n{size}_steps{steps}")
results.to_csv(OUTPUT_DIR / "dfst_universal.csv", index=False)
